# Laneless Karalakou PPO

Train a PPO ego vehicle in the custom lane-free highway-env extension using the best reward family from Karalakou et al.: **Fields + Zones + Overtake + Collision Avoidance**, plus a lateral-acceleration smoothness penalty.

The simulator still comes from `laneless highway env/lane_free_env.py`. This notebook adds only an RL reward wrapper and training/evaluation code.

If an import fails, install the dependencies in the selected notebook interpreter:

```python
# !python -m pip install -U stable-baselines3[extra] gymnasium highway-env torch numpy pandas matplotlib pygame
```

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from typing import Any, Optional

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "laneless highway env" / "lane_free_env.py").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "laneless highway env" / "lane_free_env.py").exists():
            return nested
    raise RuntimeError("Could not find the project root containing 'laneless highway env/lane_free_env.py'.")


PROJECT_ROOT = find_project_root()
LANE_FREE_DIR = PROJECT_ROOT / "laneless highway env"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "lanelessKaralakou"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

lane_free_dir_str = str(LANE_FREE_DIR)
if lane_free_dir_str not in sys.path:
    sys.path.insert(0, lane_free_dir_str)

import lane_free_env  # noqa: F401: registers lane-free-v0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Project root:", PROJECT_ROOT)
print("Lane-free env dir:", LANE_FREE_DIR)
print("Device:", DEVICE)

## Karalakou Best Reward Wrapper

The paper's best-performing reward function combines:

- desired longitudinal speed cost `cx`
- lateral target/zone cost `cy`
- potential-field collision-risk cost `cf`
- collision penalty `rc`
- lateral acceleration cost `cay`
- overtake bonus `ro`

The wrapper below replaces the environment reward during PPO training without changing the simulator source code.

In [ ]:
class KaralakouRewardWrapper(gym.Wrapper):
    """Reward wrapper for Fields + Zones + Overtake + Collision Avoidance."""

    def __init__(self, env: gym.Env, reward_config: Optional[dict[str, float]] = None) -> None:
        super().__init__(env)
        self.reward_config = {
            "epsilon_r": 0.1,
            "wx": 0.35,
            "wy": 0.65,
            "wf": 1.0,
            "collision_penalty": -2.5,
            "way": 0.25,
            "overtake_bonus": 2.0,
            "timegap": 0.7,
            "ego_desired_speed": 20.0,
            "field_magnitude": 0.5,
            "field_px": 2.0,
            "field_py": 2.0,
            "field_pt": 1.0,
            "zone_margin": 0.15,
        }
        if reward_config:
            self.reward_config.update(reward_config)

    @property
    def base_env(self):
        return self.env.unwrapped

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.base_env.vehicle.desired_speed = float(self.reward_config["ego_desired_speed"])
        return self.base_env._observe(), info

    def step(self, action):
        previous_dx = self._relative_dx_by_vehicle()
        obs, _, terminated, truncated, info = self.env.step(action)
        reward, components = self._karalakou_reward(previous_dx)

        info = dict(info)
        info.update({f"karalakou_{key}": value for key, value in components.items()})
        return obs, reward, terminated, truncated, info

    def _relative_dx_by_vehicle(self) -> dict[int, float]:
        base = self.base_env
        ego = base.vehicle
        return {
            id(vehicle): base._signed_distance(ego.position[0], vehicle.position[0])
            for vehicle in base.road.vehicles
            if vehicle is not ego
        }

    def _karalakou_reward(self, previous_dx: dict[int, float]) -> tuple[float, dict[str, float]]:
        base = self.base_env
        ego = base.vehicle
        cfg = self.reward_config

        target_y, target_speed, zone_found = self._lateral_target_and_speed()
        cx = abs(ego.vx - target_speed) / max(target_speed, 1e-6)
        cy = abs(float(ego.position[1]) - target_y) / max(float(base.config["road_width"]), 1e-6)
        cf = self._potential_field_cost()
        cay = self._lateral_acceleration_cost()
        overtakes = self._overtake_count(previous_dx)

        denom = cfg["epsilon_r"] + cfg["wx"] * cx + cfg["wy"] * cy + cfg["wf"] * cf + cfg["way"] * cay
        reward = cfg["epsilon_r"] / max(denom, 1e-9)
        if bool(base._last_ego_collision):
            reward += cfg["collision_penalty"]
        elif overtakes > 0:
            reward += cfg["overtake_bonus"] * min(overtakes, 1)

        components = {
            "reward": float(reward),
            "cx": float(cx),
            "cy": float(cy),
            "cf": float(cf),
            "cay": float(cay),
            "ay": float(self._ego_lateral_acceleration()),
            "target_y": float(target_y),
            "target_speed": float(target_speed),
            "zone_found": float(zone_found),
            "overtakes": float(overtakes),
            "ego_collision": float(base._last_ego_collision),
        }
        return float(reward), components

    def _ego_lateral_acceleration(self) -> float:
        base = self.base_env
        if len(base._last_accelerations) == 0:
            return 0.0
        return float(base._last_accelerations[0, 1])

    def _lateral_acceleration_cost(self) -> float:
        base = self.base_env
        bounds = base.config["bounds"]
        ay_scale = max(abs(float(bounds["ay_min"])), abs(float(bounds["ay_max"])), 1e-6)
        return float(min(abs(self._ego_lateral_acceleration()) / ay_scale, 1.0))

    def _potential_field_cost(self) -> float:
        base = self.base_env
        ego = base.vehicle
        cfg = self.reward_config
        sensing_range = float(base.config["sensing_range"])
        total_cost = 0.0

        for vehicle in base.road.vehicles:
            if vehicle is ego:
                continue
            dx = base._signed_distance(ego.position[0], vehicle.position[0])
            if abs(dx) > sensing_range:
                continue
            dy = float(vehicle.position[1] - ego.position[1])
            critical_a = max(0.5 * (ego.length + vehicle.length), 1e-3)
            critical_b = max(0.5 * (ego.width + vehicle.width), 1e-3)
            broad_a = critical_a + cfg["timegap"] * max(ego.vx - vehicle.vx, 0.0)
            broad_b = critical_b + cfg["timegap"] * abs(ego.vy - vehicle.vy)

            total_cost += self._ellipsoid(dx, dy, critical_a, critical_b)
            total_cost += self._ellipsoid(dx, dy, broad_a, broad_b)

        return float(min(total_cost, 1.0))

    def _ellipsoid(self, dx: float, dy: float, a: float, b: float) -> float:
        cfg = self.reward_config
        scaled = (abs(dx) / max(a, 1e-6)) ** cfg["field_px"] + (abs(dy) / max(b, 1e-6)) ** cfg["field_py"] + 1.0
        return float(cfg["field_magnitude"] / (scaled ** cfg["field_pt"]))

    def _lateral_target_and_speed(self) -> tuple[float, float, bool]:
        base = self.base_env
        ego = base.vehicle
        road_width = float(base.config["road_width"])
        sensing_range = float(base.config["sensing_range"])
        cfg = self.reward_config
        scan_distance = min(max(ego.vx * cfg["timegap"], 1.0), sensing_range)
        blockers: list[tuple[float, Any]] = []

        for vehicle in base.road.vehicles:
            if vehicle is ego:
                continue
            dx = base._forward_distance(ego.position[0], vehicle.position[0])
            if 0.0 < dx < scan_distance:
                blockers.append((dx, vehicle))

        if not blockers:
            return road_width / 2.0, ego.desired_speed, True

        y_min = ego.width / 2.0
        y_max = road_width - ego.width / 2.0
        free_intervals = [(y_min, y_max)]

        for _, vehicle in blockers:
            blocked_low = float(vehicle.position[1] - 0.5 * (ego.width + vehicle.width) - cfg["zone_margin"])
            blocked_high = float(vehicle.position[1] + 0.5 * (ego.width + vehicle.width) + cfg["zone_margin"])
            blocked_low = max(blocked_low, y_min)
            blocked_high = min(blocked_high, y_max)
            free_intervals = self._subtract_interval(free_intervals, blocked_low, blocked_high)

        free_intervals = [(lo, hi) for lo, hi in free_intervals if hi - lo > 0.05]
        if free_intervals:
            ego_y = float(ego.position[1])

            def interval_distance(interval: tuple[float, float]) -> float:
                lo, hi = interval
                if lo <= ego_y <= hi:
                    return 0.0
                return min(abs(ego_y - lo), abs(ego_y - hi))

            best_low, best_high = min(free_intervals, key=interval_distance)
            return 0.5 * (best_low + best_high), ego.desired_speed, True

        slowest_blocker = min((vehicle for _, vehicle in blockers), key=lambda vehicle: vehicle.vx)
        fastest_blocker = max((vehicle for _, vehicle in blockers), key=lambda vehicle: vehicle.vx)
        target_y = float(np.clip(fastest_blocker.position[1], y_min, y_max))
        target_speed = min(ego.desired_speed, slowest_blocker.vx)
        return target_y, target_speed, False

    @staticmethod
    def _subtract_interval(intervals: list[tuple[float, float]], low: float, high: float) -> list[tuple[float, float]]:
        if high <= low:
            return intervals
        remaining: list[tuple[float, float]] = []
        for interval_low, interval_high in intervals:
            if high <= interval_low or low >= interval_high:
                remaining.append((interval_low, interval_high))
                continue
            if low > interval_low:
                remaining.append((interval_low, low))
            if high < interval_high:
                remaining.append((high, interval_high))
        return remaining

    def _overtake_count(self, previous_dx: dict[int, float]) -> int:
        base = self.base_env
        ego = base.vehicle
        sensing_range = float(base.config["sensing_range"])
        overtakes = 0
        for vehicle in base.road.vehicles:
            if vehicle is ego:
                continue
            old_dx = previous_dx.get(id(vehicle))
            if old_dx is None:
                continue
            new_dx = base._signed_distance(ego.position[0], vehicle.position[0])
            if 0.0 < old_dx < sensing_range and new_dx < -0.5 * ego.length:
                overtakes += 1
        return overtakes

## Experiment Config

These values mirror the paper's simulation setup as closely as the current lane-free simulator allows: 500 m ring road, 10.2 m width, 35 vehicles, 200 s episodes, 0.25 s step interval, 5 observed neighbors, and 80 m sensing.

In [ ]:
ENV_CONFIG = {
    "road_length": 500.0,
    "road_width": 10.2,
    "dt": 0.25,
    "simulation_frequency": 4,
    "policy_frequency": 4,
    "vehicles_count": 35,
    "neighbors_count": 5,
    "sensing_range": 80.0,
    "episode_steps": 800,
    "duration": 800,
    "gamma_nudge": 0.0,
    "ego_controlled": True,
    "desired_speed_range": [18.0, 22.0],
    "initial_speed_fraction_range": [0.75, 1.0],
    "screen_width": 900,
    "screen_height": 260,
    "real_time_rendering": True,
}

REWARD_CONFIG = {
    "ego_desired_speed": 20.0,
    "epsilon_r": 0.1,
    "wx": 0.35,
    "wy": 0.65,
    "wf": 1.0,
    "collision_penalty": -2.5,
    "way": 0.25,
    "overtake_bonus": 2.0,
    "timegap": 0.7,
}

TOTAL_TIMESTEPS = 50_000
EVAL_EVERY = 5_000
EVAL_EPISODES = 5
FINAL_EVAL_EPISODES = 20
SEED = 7

MODEL_PATH = ARTIFACT_DIR / "ppo_laneless_karalakou.zip"
HISTORY_PATH = ARTIFACT_DIR / "ppo_laneless_karalakou_eval_history.csv"
PLOT_PATH = ARTIFACT_DIR / "ppo_laneless_karalakou_metrics.png"

In [ ]:
def make_single_env(seed: int = SEED, render_mode: Optional[str] = None) -> gym.Env:
    env = gym.make("lane-free-v0", render_mode=render_mode, config=ENV_CONFIG)
    env = KaralakouRewardWrapper(env, reward_config=REWARD_CONFIG)
    env = Monitor(env)
    env.reset(seed=seed)
    return env


def make_training_env(seed: int = SEED) -> DummyVecEnv:
    def _init() -> gym.Env:
        return make_single_env(seed=seed, render_mode=None)

    return DummyVecEnv([_init])


smoke_env = make_single_env(seed=SEED)
obs, info = smoke_env.reset(seed=SEED)
print("Observation shape:", obs.shape)
print("Action space:", smoke_env.action_space)
print("Initial ego desired speed:", smoke_env.unwrapped.vehicle.desired_speed)
smoke_env.close()

## Evaluation Helpers

The callback evaluates the current PPO policy every few thousand training steps and records:

- mean signed speed deviation `vx - desired_speed`
- mean absolute speed deviation `abs(vx - desired_speed)`
- ego collision count
- total traffic collision events

In [ ]:
def evaluate_policy_with_metrics(
    model: PPO,
    episodes: int = EVAL_EPISODES,
    seed: int = SEED + 1_000,
    render_mode: Optional[str] = None,
    deterministic: bool = True,
) -> pd.DataFrame:
    rows: list[dict[str, float]] = []
    for episode in range(episodes):
        env = make_single_env(seed=seed + episode, render_mode=render_mode)
        obs, _ = env.reset(seed=seed + episode)
        done = False
        step_count = 0
        rewards = []
        signed_deviations = []
        abs_deviations = []
        speeds = []
        ego_collisions = 0
        all_collision_events = 0

        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            base = env.unwrapped
            desired = float(base.vehicle.desired_speed)
            speed = float(base.vehicle.vx)
            deviation = speed - desired

            rewards.append(float(reward))
            signed_deviations.append(deviation)
            abs_deviations.append(abs(deviation))
            speeds.append(speed)
            all_collision_events += int(info.get("collisions", 0))
            if bool(info.get("ego_collision", False)):
                ego_collisions += 1

            step_count += 1
            done = bool(terminated or truncated)

        rows.append(
            {
                "episode": episode,
                "steps": step_count,
                "return": float(np.sum(rewards)),
                "mean_speed": float(np.mean(speeds)) if speeds else 0.0,
                "mean_signed_speed_deviation": float(np.mean(signed_deviations)) if signed_deviations else 0.0,
                "mean_abs_speed_deviation": float(np.mean(abs_deviations)) if abs_deviations else 0.0,
                "ego_collisions": float(ego_collisions),
                "total_collision_events": float(all_collision_events),
            }
        )
        env.close()
    return pd.DataFrame(rows)


class EvalMetricsCallback(BaseCallback):
    def __init__(self, eval_freq: int, n_eval_episodes: int, seed: int, verbose: int = 1) -> None:
        super().__init__(verbose=verbose)
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.seed = int(seed)
        self.records: list[dict[str, float]] = []
        self._last_eval_step = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last_eval_step < self.eval_freq:
            return True
        self._last_eval_step = self.num_timesteps
        metrics = evaluate_policy_with_metrics(
            self.model,
            episodes=self.n_eval_episodes,
            seed=self.seed + self.num_timesteps,
            deterministic=True,
        )
        row = {
            "timesteps": float(self.num_timesteps),
            "return": float(metrics["return"].mean()),
            "mean_signed_speed_deviation": float(metrics["mean_signed_speed_deviation"].mean()),
            "mean_abs_speed_deviation": float(metrics["mean_abs_speed_deviation"].mean()),
            "ego_collisions": float(metrics["ego_collisions"].mean()),
            "total_collision_events": float(metrics["total_collision_events"].mean()),
        }
        self.records.append(row)
        if self.verbose:
            print(
                f"steps={self.num_timesteps:,} | "
                f"abs dev={row['mean_abs_speed_deviation']:.3f} m/s | "
                f"ego collisions={row['ego_collisions']:.2f} | "
                f"traffic collisions={row['total_collision_events']:.2f}"
            )
        return True

## Train PPO

Increase `TOTAL_TIMESTEPS` for a stronger policy. Start with 50k to verify the pipeline, then move to 200k+ for a better experiment.

In [ ]:
train_env = make_training_env(seed=SEED)
eval_callback = EvalMetricsCallback(eval_freq=EVAL_EVERY, n_eval_episodes=EVAL_EPISODES, seed=SEED + 10_000)

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.98,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs={"net_arch": [256, 128]},
    tensorboard_log=str(ARTIFACT_DIR / "tensorboard"),
    verbose=1,
    seed=SEED,
    device=DEVICE,
)

start_time = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
elapsed = time.time() - start_time

model.save(str(MODEL_PATH))
train_env.close()

eval_history = pd.DataFrame(eval_callback.records)
eval_history.to_csv(HISTORY_PATH, index=False)
print(f"Saved model to {MODEL_PATH}")
print(f"Saved eval history to {HISTORY_PATH}")
print(f"Training time: {elapsed / 60:.1f} min")
display(eval_history)

## Plot Training Results

In [ ]:
if "eval_history" not in globals() or eval_history.empty:
    eval_history = pd.read_csv(HISTORY_PATH)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(eval_history["timesteps"], eval_history["mean_abs_speed_deviation"], marker="o", label="abs(vx - vd)")
axes[0].plot(eval_history["timesteps"], eval_history["mean_signed_speed_deviation"].abs(), marker="s", label="abs(mean(vx - vd))")
axes[0].set_title("Deviation from desired speed")
axes[0].set_xlabel("Training timesteps")
axes[0].set_ylabel("Speed deviation (m/s)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(eval_history["timesteps"], eval_history["ego_collisions"], marker="o", label="ego collisions")
axes[1].plot(eval_history["timesteps"], eval_history["total_collision_events"], marker="s", label="all traffic collision events")
axes[1].set_title("Collisions per evaluation episode")
axes[1].set_xlabel("Training timesteps")
axes[1].set_ylabel("Collision count")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=160)
plt.show()
print("Saved plot to", PLOT_PATH)

## Final Evaluation

In [ ]:
trained_model = PPO.load(str(MODEL_PATH), device=DEVICE)
final_metrics = evaluate_policy_with_metrics(
    trained_model,
    episodes=FINAL_EVAL_EPISODES,
    seed=SEED + 50_000,
    deterministic=True,
)
display(final_metrics)
display(final_metrics.drop(columns=["episode"]).mean().to_frame("mean"))

## Render The Trained PPO Policy

Set `RUN_RENDER = True` when you want a live highway-env render window.

In [ ]:
RUN_RENDER = False
RENDER_STEPS = 1_000

if RUN_RENDER:
    render_env = make_single_env(seed=SEED + 99_000, render_mode="human")
    obs, _ = render_env.reset(seed=SEED + 99_000)
    for _ in range(RENDER_STEPS):
        action, _ = trained_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(action)
        if terminated or truncated:
            obs, _ = render_env.reset()
    render_env.close()